# Setup of an apo protein

Goal of this notebook is to:
1. Split the pdb into protein and ligand pieces
2. Use PDBFixer to fix common problems in the protein structure.
3. Output a cleaned and restored structure for the next step

I recommend trying all the steps in this notebook using the provided pdb code first, then save the notebook to a unique name, and try repeating the steps for your protein system.


In [ ]:
from pathlib import Path

import openmm.app as app
import openmm.unit as unit

from pdbfixer.pdbfixer import PDBFixer
import mdtraj as md
import datamol as dm

from pymol import cmd

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import inchi

In [ ]:
id = '4trn'

# make a subdirectory for storing our structures
Path('pdbs').mkdir(exist_ok=True)

cmd.fetch(id, type="pdb", path="./pdbs/") 

## Print out some pdb file details

If you structure has co-factors, crystallization compounds, or bound inhibitors, they will be listed in the Heteroatoms section

In [ ]:
# Header and file size
print("\n\n","="*30, "Header Information", "="*30)
with open(f"./pdbs/{id}.pdb") as pdb_file:
    show = 10  # Lines to show
    for lc, line in enumerate(pdb_file):
        # Print the first lines of the file
        if lc < show:
            print(line.strip())
print("\nTotal line count: ", lc)

# Heteroatoms (non protein)
print("\n\n","="*30, "Heteroatom Summary", "="*30)
with open(f"./pdbs/{id}.pdb") as pdb_file:
    for line in pdb_file:
        if line.startswith(("HETNAM", "FORMUL")):
            print(line.strip())

# Additional Bond linkages outside of normal templates
# 
print("\n\n","="*30, "Bonds Summary", "="*30)
with open(f"./pdbs/{id}.pdb") as pdb_file:
    for line in pdb_file:
        if line.startswith(("LINK", "SSBOND")):
            print(line.strip())

## Splitting protein and ligands

Looking at the Heteroatom Summary for `4trn` (anything not the protein itself), you can see:

1. NAD - Co-factor for this enzyme
2. MPD - Crystallization Compound
3. DMS - Crystallization Compound
4. HOH - Crystal Waters

Generally only the protein and the NAD co-factor would be biologically relevent for this structure.  The waters may or may not be relevent, but are usually removed in most protein setup workflows.  They are added back later when we immerse the protein in a periodic water box.

We'll use mdtraj to split the protein and ligand and then focus on fixing any protein issues using PDBfixer.

Make sure to change the ligand resname variable below to match any ligand you want to keep.  If you have multiple ligands to keep, copy those 4 lines and edit the `ligand_resname` field for each ligand.

In [ ]:
# pdb splitting
traj = md.load(f"./pdbs/{id}.pdb") 

# protein
protein = traj.atom_slice(traj.top.select("protein"))
protein.save_pdb(f"./pdbs/{id}_protein.pdb")

# Specific ligand (by resname) - repeat this section for any additional ligands
ligand_resname = "NAD"
ligand_atoms = traj.top.select(f"resname {ligand_resname}")
ligand = traj.atom_slice(ligand_atoms)
ligand.save_pdb(f"./pdbs/{id}_{ligand_resname}.pdb")

## PDBFixer to patch missing residues and atoms

PDBFixer can restore missing atoms and residues in an x-ray structure and properly set N and C termini residues.

Although it will happily patch flexible loops of any length, the longer the loop, the less likely the starting geometry is reasonable and it may take significant equilibration time to find a reasonable conformation.  The longer the stretch of amino acids missing, the more consideration should be given to openMM's modeller function, or even more sophisticated tools like MODELLER (yes same name, but it's an external tool that does simulated annealing to model loops).

In the below example, the number of atoms patched 

In [ ]:
# standard pdbfixer pass
# restore missing residues and atoms
# Note that the fix order matters - find missing atoms must be last

fixer = PDBFixer(f'./pdbs/{id}.pdb')

fixer.findMissingResidues()
print("**** Missing residue summary ****\n")
for (chain_idx, start), residues in fixer.missingResidues.items():
    print(f"CHAIN {chain_idx} : {start+1}-{start+len(residues)} : {residues}")

fixer.findNonstandardResidues()
print("\n**** Non-standard residues ****\n")
if fixer.nonstandardResidues == []:
    print("No non-standard residues found")
else:
    print(fixer.nonstandardResidues)

fixer.replaceNonstandardResidues()
fixer.removeHeterogens(False)

fixer.findMissingAtoms()
print("\n**** Total number of heavy atoms missing ****\n")
print(f"N = {len(fixer.missingAtoms)}")

fixer.addMissingAtoms()
fixer.addMissingHydrogens(7)

app.PDBFile.writeFile(fixer.topology, fixer.positions, open(f'./pdbs/{id}_fixer.pdb', 'w'))

## Checking processed protein structure

You should now have a clean protein structure (`{id}_fixer.pdb`)

Let's read it and check for any issues.  If these tests pass, you should be ready to create a system in openmm for minimization and dynamics simulations.

In [ ]:
test = PDBFixer(f'./pdbs/{id}_fixer.pdb')

# Verify Changes
print("\nChecking structure after changes")

test.findMissingResidues()
if hasattr(test, "missingResidues"):
    for (chain_idx, start), residues in test.missingResidues.items():
        print(f"CHAIN {chain_idx} : {start+1}-{start+len(residues)} : {residues}")
else:
    print("No missing residues")
    
test.findNonstandardResidues()
if hasattr(test, "nonstandardResidues"):
    print(test.nonstandardResidues)
else:
    print("No non-standard residues found")

test.findMissingAtoms()
if hasattr(test, "missingAtoms"):
    print(f"# missing atoms = {len(test.missingAtoms)}")
    print(test.missingAtoms)
else:
    print("No missing atoms")
 


## Ligand processing (Skip this step on your first pass)

In the pdb entry page at https://www.rcsb.org/, if you go down to the ligands section and click the 3 letter code for your ligand, it will bring you to a separate page that has a section of chemical descriptors.  One of those is the canonical smiles.

In the example below for NAD+, it lists:

`NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@](O)(=O)OC[C@H]3O[C@H]([C@H](O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O`

However, closer inspection shows one phosphate oxygen protonated, which is not the typical form expected at pH=7.  A manual edit was made to the smiles string to change `[P@](O)` to `[P@]([O-])`, yielding:

`NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@]([O-])(=O)OC[C@H]3O[C@H]([C@H](O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O`

Using this we'll build a template molecule from the smiles string in rdkit, then merge the coordinates from the ligand pdb file using the function:

`AllChem.AssignBondOrdersFromTemplate( template, coordinates )` 

Make sure to check:
* Total charge
* Protonation state of any titratable groups
* sdf structure overlaid on ligand.pdb source and verify geometry and protonation

This step is critical before trying to parameterize this ligand for simulations.

In [ ]:
# canonical smiles for ligand
# under the ligands section, click the 3 letter code link in your source pdb
# brings you to the ligands page (ex. https://www.rcsb.org/ligand/NAD )
# there look for canonical smiles, which saves stereochemistry and formal charges unambiguously
# MANUAL EDIT NEEDED: they have one phosphate oxygen protonated, but we generally want it deprotonated
# So [P@](O) has been changed to [P@]([O-])
# rdkit then constructs the reference molecule from the smiles string and assigns coordinates from the pdb file

smiles = "NC(=O)c1ccc[n+](c1)[C@@H]2O[C@H](CO[P]([O-])(=O)O[P@]([O-])(=O)OC[C@H]3O[C@H]([C@H](O)[C@@H]3O)n4cnc5c(N)ncnc45)[C@@H](O)[C@H]2O"
template = Chem.MolFromSmiles(smiles)

ligand = Chem.MolFromPDBFile(f"./pdbs/{id}_{ligand_resname}.pdb", removeHs=False )
ligand = AllChem.AssignBondOrdersFromTemplate( template, ligand )
ligand = Chem.AddHs(ligand, addCoords=True)

# Additional clean-up and standardization
Chem.AssignStereochemistry(ligand, force=True)
Chem.SanitizeMol(ligand)

# write sdf file
Chem.SDWriter(f"./pdbs/{id}_{ligand_resname}.sdf").write(ligand)

# final checks - compare template and standardized ligand
print("Template atoms:", template.GetNumAtoms())
print("Template heavy atoms:", template.GetNumHeavyAtoms())
print("Ligand atoms:", ligand.GetNumAtoms())
print("Ligand heavy atoms:", ligand.GetNumHeavyAtoms())

charge = sum([ atom.GetFormalCharge() for atom in ligand.GetAtoms() ])
print("Total Formal charge =", charge)

# Writing ligand metadata - standard chemical identifiers
smiles = Chem.MolToSmiles(ligand, canonical=True, isomericSmiles=True)
inchi_str = inchi.MolToInchi(ligand)
inchikey = inchi.MolToInchiKey(ligand)

print(f"SMILES: {smiles} \nInChI: {inchi_str} \nInChIKey: {inchikey}")

Path(f"./pdbs/{id}_{ligand_resname}_metadata.txt").write_text(
     f"SMILES:   {smiles}\n"
     f"InChI:    {inchi_str}\n"
     f"InChIKey: {inchikey}\n"
)


svg = dm.viz.to_image(ligand, outfile=f"./pdbs/{id}_{ligand_resname}.svg", use_svg=True, indices=True)
svg